# Driver Drowsiness: Eye-State Model (Google Colab)

Trains a MobileNetV2 open/closed eye classifier and exports `eye_state.tflite` for the Raspberry Pi.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

What makes it accurate and trustworthy:
- **Subject-wise split**: train / val / test contain *different people*, so test accuracy is honest (no frame leakage).
- Heavy augmentation (brightness, gamma, blur, noise, rotation, shift) so it survives real webcams.
- Two-stage transfer learning (head, then fine-tune) with cosine LR decay.
- Threshold tuning on validation data, final numbers on unseen test subjects.
- TFLite export is re-tested on the test set.

Sections: 1 Setup, 2 Config, 3 Dataset, 4 Index and split, 5 Input pipeline, 6 Model, 7 Training, 8 Evaluation, 9 Export, 10 Download

## 1. Setup

In [ ]:
%pip install -q kagglehub

import json, math, pathlib, random, re, time
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
if not gpus:
    print("WARNING: no GPU. Use Runtime -> Change runtime type -> T4 GPU")

## 2. Config
All tunable values in one place.

In [ ]:
@dataclass
class Cfg:
    img_size: int = 96            # must match ModelConfig.img_size on the Pi
    alpha: float = 0.5            # MobileNetV2 width: 0.35 (fastest) / 0.5 / 0.75
    weights: Optional[str] = "imagenet"
    batch: int = 128
    epochs_head: int = 3
    epochs_ft: int = 15
    lr_head: float = 1e-3
    lr_ft: float = 2e-4
    dropout: float = 0.3
    label_smoothing: float = 0.05
    n_val_subjects: int = 5
    n_test_subjects: int = 5
    seed: int = 42
    quantization: str = "dynamic"   # "dynamic" | "float16" | "none"
    out_dir: str = "/content/output"

CFG = Cfg()
random.seed(CFG.seed); np.random.seed(CFG.seed); tf.keras.utils.set_random_seed(CFG.seed)

## 3. Dataset (MRL Eye Dataset)
Downloads from Kaggle with `kagglehub`. If that fails, upload the dataset zip manually (instructions in the comments).

In [ ]:
import kagglehub

# Option A (automatic)
DATA_ROOT = kagglehub.dataset_download("imadeddinedjerarda/mrl-eye-dataset")

# Option B (manual): download the zip from kaggle.com/datasets/imadeddinedjerarda/mrl-eye-dataset,
# then run these lines instead:
#   from google.colab import files; files.upload()          # choose the zip
#   !unzip -q *.zip -d /content/mrl
#   DATA_ROOT = "/content/mrl"
print("Dataset at:", DATA_ROOT)

## 4. Index images and split by subject
MRL filenames encode `subject_image_gender_glasses_eyestate_...` (eye state 0 = closed, 1 = open). Label used here: **closed = 1, open = 0** (model outputs P(closed)).

In [ ]:
class EyeIndex:
    MRL = re.compile(r"^(s\d+)_\d+_\d+_\d+_([01])_")
    CLOSED_WORDS = ("closed", "close", "sleepy", "drows")
    OPEN_WORDS = ("open", "awake")
    EXTS = (".png", ".jpg", ".jpeg")

    def __init__(self, root, cfg: Cfg):
        self.root, self.cfg = pathlib.Path(root), cfg

    def _images(self):
        return [p for p in self.root.rglob("*") if p.suffix.lower() in self.EXTS]

    def _scan_mrl(self, files):
        rows = []
        for p in files:
            m = self.MRL.match(p.name)
            if m:
                rows.append((str(p), m.group(1), 1 if m.group(2) == "0" else 0))
        return rows

    def _scan_folders(self, files):
        """Fallback for forks that only keep open/closed folders (no subject ids)."""
        rows = []
        for i, p in enumerate(sorted(files)):
            rel = str(p.parent.relative_to(self.root)).lower()
            if any(w in rel for w in self.CLOSED_WORDS):
                label = 1
            elif any(w in rel for w in self.OPEN_WORDS):
                label = 0
            else:
                continue
            rows.append((str(p), f"pseudo{i % 40}", label))   # NOT a true subject split
        return rows

    def build(self):
        files = self._images()
        rows = self._scan_mrl(files)
        subject_wise = bool(rows)
        if not rows:
            print("MRL filename pattern not found -> using folder names (split is NOT subject-wise).")
            rows = self._scan_folders(files)
        if not rows:
            raise RuntimeError(f"No labelled eye images found under {self.root}")

        df = pd.DataFrame(rows, columns=["path", "subject", "label"])
        print(f"{len(df)} images | {df.subject.nunique()} subjects | closed fraction {df.label.mean():.2f}"
              f" | subject-wise labels: {subject_wise}")
        return self._split(df)

    def _split(self, df):
        c = self.cfg
        subjects = sorted(df.subject.unique())
        random.Random(c.seed).shuffle(subjects)
        test_s = set(subjects[: c.n_test_subjects])
        val_s = set(subjects[c.n_test_subjects: c.n_test_subjects + c.n_val_subjects])
        parts = {
            "train": df[~df.subject.isin(test_s | val_s)],
            "val": df[df.subject.isin(val_s)],
            "test": df[df.subject.isin(test_s)],
        }
        for name, d in parts.items():
            if d.label.nunique() < 2:
                raise RuntimeError(f"{name} split has a single class; change the seed or subject counts")
            print(f"  {name:5s}: {len(d):6d} images, {d.subject.nunique():2d} subjects, closed {d.label.mean():.2f}")
        return parts


splits = EyeIndex(DATA_ROOT, CFG).build()

## 5. Input pipeline
Grayscale, resized, cached as uint8 in RAM. Augmentation runs per image on the training set only.

In [ ]:
class DatasetBuilder:
    def __init__(self, cfg: Cfg):
        self.cfg = cfg
        self.aug = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.06),
            tf.keras.layers.RandomZoom(0.12),
            tf.keras.layers.RandomTranslation(0.08, 0.08),
            tf.keras.layers.RandomBrightness(0.35, value_range=(0, 255)),
            tf.keras.layers.RandomContrast(0.35),
        ])

    def _read(self, path, label):
        s = self.cfg.img_size
        img = tf.io.decode_image(tf.io.read_file(path), channels=1, expand_animations=False)
        img.set_shape([None, None, 1])
        img = tf.image.resize(img, [s, s])
        return tf.cast(tf.round(img), tf.uint8), label

    def _augment(self, x, y):
        s = self.cfg.img_size
        x = self.aug(tf.cast(x, tf.float32), training=True)
        k = tf.random.uniform([], 48, s + 1, tf.int32)               # random blur / low-res
        x = tf.image.resize(tf.image.resize(x, [k, k]), [s, s])
        gamma = tf.random.uniform([], 0.6, 1.6)                      # exposure variation
        x = 255.0 * tf.pow(tf.clip_by_value(x, 0.0, 255.0) / 255.0, gamma)
        x = x + tf.random.normal(tf.shape(x), stddev=tf.random.uniform([], 0.0, 10.0))  # sensor noise
        return tf.clip_by_value(x, 0.0, 255.0), y

    @staticmethod
    def _finalize(x, y):
        x = tf.image.grayscale_to_rgb(tf.cast(x, tf.float32))        # (S,S,3), range 0-255
        return x, tf.expand_dims(y, -1)

    def build(self, df: pd.DataFrame, training: bool):
        a = tf.data.AUTOTUNE
        ds = tf.data.Dataset.from_tensor_slices((df.path.values, df.label.values.astype("float32")))
        ds = ds.map(self._read, num_parallel_calls=a).cache()
        if training:
            ds = ds.shuffle(10000, seed=self.cfg.seed).map(self._augment, num_parallel_calls=a)
        return ds.map(self._finalize, num_parallel_calls=a).batch(self.cfg.batch).prefetch(a)


builder = DatasetBuilder(CFG)
train_ds = builder.build(splits["train"], training=True)
val_ds = builder.build(splits["val"], training=False)
test_ds = builder.build(splits["test"], training=False)

# sanity check: preview augmented training samples
xb, yb = next(iter(train_ds))
fig, axs = plt.subplots(2, 8, figsize=(16, 4))
for ax, img, lab in zip(axs.ravel(), xb.numpy(), yb.numpy()):
    ax.imshow(img[..., 0].astype("uint8"), cmap="gray"); ax.set_title("closed" if lab[0] else "open"); ax.axis("off")
plt.show()

## 6. Model
MobileNetV2 backbone, input 0-255 (rescaling lives inside the model), single sigmoid output = P(closed).

In [ ]:
class EyeModel:
    def __init__(self, cfg: Cfg):
        self.cfg = cfg
        s = cfg.img_size
        inp = tf.keras.Input((s, s, 3), name="eye_crop")
        x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(inp)
        self.base = tf.keras.applications.MobileNetV2(
            input_shape=(s, s, 3), alpha=cfg.alpha, include_top=False, weights=cfg.weights)
        self.base.trainable = False
        x = self.base(x, training=False)
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        x = tf.keras.layers.Dropout(cfg.dropout)(x)
        out = tf.keras.layers.Dense(1, activation="sigmoid", name="p_closed")(x)
        self.model = tf.keras.Model(inp, out, name="eye_state")

    def compile(self, lr):
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(lr),
            loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=self.cfg.label_smoothing),
            metrics=[tf.keras.metrics.BinaryAccuracy(name="acc"),
                     tf.keras.metrics.AUC(name="auc")])

    def unfreeze(self):
        self.base.trainable = True
        for layer in self.base.layers:       # keep BatchNorm statistics frozen
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False


eye = EyeModel(CFG)
eye.model.summary(line_length=90)

## 7. Training
Stage 1 trains the new head; stage 2 fine-tunes the whole backbone with a cosine learning-rate schedule.

In [ ]:
class Trainer:
    def __init__(self, cfg: Cfg, eye: EyeModel):
        self.cfg, self.eye = cfg, eye

    @staticmethod
    def class_weights(df):
        n, n1 = len(df), int(df.label.sum())
        n0 = n - n1
        return {0: n / (2 * n0), 1: n / (2 * n1)}

    def fit(self, train_ds, val_ds, train_df):
        c, m = self.cfg, self.eye.model
        cw = self.class_weights(train_df)
        print("class weights:", cw)

        print("\n== Stage 1: classifier head ==")
        self.eye.compile(c.lr_head)
        h1 = m.fit(train_ds, validation_data=val_ds, epochs=c.epochs_head, class_weight=cw)

        print("\n== Stage 2: fine-tune backbone ==")
        self.eye.unfreeze()
        steps = math.ceil(len(train_df) / c.batch) * c.epochs_ft
        lr = tf.keras.optimizers.schedules.CosineDecay(c.lr_ft, steps, alpha=0.01)
        self.eye.compile(lr)
        stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
        h2 = m.fit(train_ds, validation_data=val_ds, epochs=c.epochs_ft, class_weight=cw, callbacks=[stop])
        return h1.history, h2.history


history = Trainer(CFG, eye).fit(train_ds, val_ds, splits["train"])

fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for h, tag in zip(history, ("head", "finetune")):
    axs[0].plot(h["val_loss"], label=f"val {tag}"); axs[1].plot(h["val_acc"], label=f"val {tag}")
axs[0].set_title("val loss"); axs[1].set_title("val accuracy"); axs[0].legend(); axs[1].legend(); plt.show()

## 8. Evaluation
The decision threshold is tuned on the **validation** subjects. Final numbers are on the **test** subjects (never seen in training).

In [ ]:
class Evaluator:
    @staticmethod
    def predict(model, ds):
        return model.predict(ds, verbose=0).ravel()

    @staticmethod
    def best_threshold(y, p):
        grid = np.arange(0.20, 0.81, 0.05)
        scores = [f1_score(y, p >= t) for t in grid]
        return float(grid[int(np.argmax(scores))])

    @staticmethod
    def report(y, p, thr, title):
        pred = p >= thr
        res = {
            "threshold": round(float(thr), 2),
            "accuracy": accuracy_score(y, pred),
            "precision_closed": precision_score(y, pred),
            "recall_closed": recall_score(y, pred),
            "f1_closed": f1_score(y, pred),
            "auc": roc_auc_score(y, p),
        }
        print(f"\n{title}")
        for k, v in res.items():
            print(f"  {k:17s}: {v:.4f}")
        print("  confusion [[TN FP] [FN TP]]:\n", confusion_matrix(y, pred))
        return {k: float(v) for k, v in res.items()}


ev = Evaluator()
p_val = ev.predict(eye.model, val_ds)
p_test = ev.predict(eye.model, test_ds)
y_val, y_test = splits["val"].label.values, splits["test"].label.values

thr = ev.best_threshold(y_val, p_val)
metrics = {
    "test_at_0.5": ev.report(y_test, p_test, 0.5, "TEST (unseen subjects), threshold 0.5"),
    "test_at_tuned": ev.report(y_test, p_test, thr, f"TEST (unseen subjects), tuned threshold {thr:.2f}"),
}

## 9. Export to TFLite and re-verify

In [ ]:
class TFLiteExporter:
    def __init__(self, cfg: Cfg):
        self.cfg = cfg

    def export(self, model, path: pathlib.Path):
        conv = tf.lite.TFLiteConverter.from_keras_model(model)
        if self.cfg.quantization in ("dynamic", "float16"):
            conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if self.cfg.quantization == "float16":
            conv.target_spec.supported_types = [tf.float16]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(conv.convert())
        print(f"Saved {path}  ({path.stat().st_size / 1024:.0f} KB)")
        return path

    @staticmethod
    def verify(path: pathlib.Path, ds, max_samples=2000, thr=0.5):
        it = tf.lite.Interpreter(model_path=str(path))
        it.allocate_tensors()
        i_idx, o_idx = it.get_input_details()[0]["index"], it.get_output_details()[0]["index"]
        correct = total = 0
        t0 = time.perf_counter()
        for xb, yb in ds:
            for img, lab in zip(xb.numpy(), yb.numpy()):
                it.set_tensor(i_idx, img[None].astype(np.float32))
                it.invoke()
                correct += int((it.get_tensor(o_idx)[0][0] >= thr) == bool(lab[0]))
                total += 1
                if total >= max_samples:
                    break
            if total >= max_samples:
                break
        ms = (time.perf_counter() - t0) / total * 1000
        acc = correct / total
        print(f"TFLite test accuracy on {total} images: {acc:.4f}  ({ms:.1f} ms/img on this CPU)")
        return acc


out = pathlib.Path(CFG.out_dir)
exporter = TFLiteExporter(CFG)
tflite_path = exporter.export(eye.model, out / "eye_state.tflite")
metrics["tflite_test_accuracy"] = exporter.verify(tflite_path, test_ds, thr=0.5)

eye.model.save(out / "eye_state.keras")
(out / "metrics.json").write_text(json.dumps({**metrics, "tuned_threshold": thr, "config": CFG.__dict__}, indent=2))
print(json.dumps(metrics, indent=2))

## 10. Download
Put `eye_state.tflite` into the `models/` folder of the project on your Mac / Raspberry Pi.
If you used `alpha` or `img_size` other than the defaults, set the same `img_size` in `ModelConfig` (default 96).

In [ ]:
from google.colab import files
files.download(str(out / "eye_state.tflite"))
files.download(str(out / "metrics.json"))